In [1]:
from pathlib import Path
from typing import Dict
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModel
from config.labels import NER_LABELS
from datasets import Dataset
import json
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from pathlib import Path
from tqdm.auto import tqdm


/Users/hunkim/Library/CloudStorage/OneDrive-HarvardUniversity/Github/ac215_Spatially/llm/development_plans/NER/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from train import Trainer

ner_trainer = Trainer()
ner_trainer.train()

Some weights of BertForTokenClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using MPS device
🧩 Preparing model input data...


Preprocessing long Label Studio entries:   0%|          | 0/4 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (2896 > 512). Running this sequence through the model will result in indexing errors


This model accepts maximum 512 as an input
We have 2896 tokens in our text
We will need to chunk the text into smaller segments
✅ Created 6 chunks from 4 documents
This means we have splite the 1 whole document into 6 chunks
This model accepts maximum 512 as an input
We have 5391 tokens in our text
We will need to chunk the text into smaller segments
✅ Created 17 chunks from 4 documents
This means we have splite the 1 whole document into 17 chunks


Preprocessing long Label Studio entries: 100%|██████████| 4/4 [00:00<00:00, 32.18it/s]


...

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.60it/s]


Epoch 3/3


Training: 100%|██████████| 4/4 [00:01<00:00,  2.67it/s]


Average loss: 0.0825


Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.55it/s]


In [2]:
from config.labels import NER_LABELS

In [3]:

model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
ner_model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(NER_LABELS)  # define this later
)
bert_model = AutoModel.from_pretrained(
    model_name
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Demonstrate a forward pass through the BERT model with dummy input
sample_text = "The quick brown fox jumps over the lazy dog."
inputs = tokenizer(sample_text, return_tensors="pt")
with torch.no_grad():
    outputs = bert_model(**inputs)
# Print the shape of the last hidden state
print("Last hidden state shape:", outputs.last_hidden_state.shape)

print(f"The embedding dimension would be: ", outputs.last_hidden_state.shape[-1])


Last hidden state shape: torch.Size([1, 14, 768])
The embedding dimension would be:  768


In [5]:
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

['[CLS]', 'the', 'quick', 'brown', 'fox', 'jump', '##s', 'over', 'the', 'laz', '##y', 'dog', '.', '[SEP]']


In [6]:
label_studio_data_path = "tmp/sample-bpda-data.json"
with open(label_studio_data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(data[0])

{'id': 248, 'annotations': [{'id': 4, 'completed_by': 1, 'result': [{'id': 'UpvLJRCBWm', 'type': 'labels', 'value': {'end': 819, 'text': 'Article 80E', 'start': 808, 'labels': ['ARTICLE_REFERENCE']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': 'i9aH8BnOeQ', 'type': 'labels', 'value': {'end': 1478, 'text': 'consists of six (6) parcels which contain approximately\\n14,678 square feet', 'start': 1403, 'labels': ['CONSTRUCTION_DETAILS']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': '9Xcr_XWsDC', 'type': 'labels', 'value': {'end': 1705, 'text': 'The Proposed Project is ideally situated walking\\ndistance from the MBTA’s Roxbury Crossing train station as well as various bus\\nroutes', 'start': 1569, 'labels': ['LOCATION_CONTEXT']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': 'qpQbSR7R5V', 'type': 'labels', 'value': {'end': 2058, 'text': 'six-story building', 'start': 2040, 'labels': ['CONSTRUCTION_DETAILS']}, 'origin'

In [7]:
# get the offsets
encoding = tokenizer(
    sample_text,
    return_offsets_mapping=True,  # offset_mapping gives you the start and end tokens for each word
    add_special_tokens=False,
)
offsets = encoding[
    "offset_mapping"
]
offsets

[(0, 3),
 (4, 9),
 (10, 15),
 (16, 19),
 (20, 24),
 (24, 25),
 (26, 30),
 (31, 34),
 (35, 38),
 (38, 39),
 (40, 43),
 (43, 44)]

In [8]:
label_to_id = {v: i for i, v in enumerate(NER_LABELS)}
id_to_label = {i: v for v, i in label_to_id.items()}
NULL_LABEL = "O"
label_to_id[NULL_LABEL] = len(label_to_id)

In [9]:
def preprocess_label_studio_data_for_long_text(
    json_path: Path = Path("tmp/sample-bpda-data.json")
) -> Dataset:
    global tokenizer
    """
    Load a Label Studio JSON file and convert it to a Hugging Face Dataset.
    Handles long documents by smartly chunking them into multiple smaller segments
    that respect entity boundaries and token limits.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed_examples = []

    for entry in tqdm(data, desc="Preprocessing long Label Studio entries"):
        text = entry["data"].get("text", "")
        if not text.strip():
            continue

        # Extract entity spans
        entities = []
        try:
            results = entry["annotations"][0]["result"]
        except (KeyError, IndexError):
            continue

        for r in results:
            if "value" not in r or "labels" not in r["value"]:
                continue
            val = r["value"]
            entities.append(
                {
                    "start": val["start"],
                    "end": val["end"],
                    # we assume that there is only one label per entity
                    "label": val["labels"][0],
                }
            )

        # Tokenize entire text once to get offsets
        encoding = tokenizer(
            text,
            return_offsets_mapping=True,
            add_special_tokens=False,
            truncation=False,
        )
        offsets = encoding["offset_mapping"]
        num_tokens = len(offsets)
        token_spans = [(s, e) for (s, e) in offsets if s is not None]

        max_tokens = tokenizer.model_max_length
        chunk_start_tok = 0
        print(f"This model accepts maximum {max_tokens} as an input")
        print(f"We have {num_tokens} tokens in our text")
        if num_tokens > max_tokens:
            print(f"We will need to chunk the text into smaller segments")

        while chunk_start_tok < num_tokens:
            # reserve room for [CLS] and [SEP]
            chunk_end_tok = min(chunk_start_tok + max_tokens - 2, num_tokens)
            chunk_start_char = token_spans[chunk_start_tok][0]
            chunk_end_char = (
                token_spans[chunk_end_tok - 1][1]
                if chunk_end_tok <= num_tokens
                else len(text)
            )

            # extend if an entity crosses the chunk boundary
            for ent in entities:
                if ent["start"] < chunk_end_char < ent["end"]:
                    chunk_end_char = ent["end"]
                    for i, (s, e) in enumerate(token_spans):
                        if e >= chunk_end_char:
                            chunk_end_tok = i + 1
                            break

            # extract the chunk text
            chunk_text = text[chunk_start_char:chunk_end_char]

            # adjust entity spans relative to this chunk
            chunk_entities = []
            for ent in entities:
                if (
                    ent["start"] >= chunk_start_char
                    and ent["end"] <= chunk_end_char
                ):
                    chunk_entities.append(
                        {
                            "start": ent["start"] - chunk_start_char,
                            "end": ent["end"] - chunk_start_char,
                            "label": ent["label"],
                        }
                    )

            if chunk_text.strip():
                processed_examples.append(
                    {"text": chunk_text, "entities": chunk_entities}
                )

            chunk_start_tok = chunk_end_tok
        print(
            f"✅ Created {len(processed_examples)} chunks from {len(data)} documents"
        )
        print(
            f"This means we have splite the 1 whole document into {len(processed_examples)} chunks"
        )

    print(
        f"✅ In total, we have created {len(processed_examples)} chunks from {len(data)} documents"
    )
    return Dataset.from_list(processed_examples)

def process_one_label_studio_entry(entry: Dict) -> Dict:
    """
    Process one pre-chunked example from preprocess_label_studio_data_for_long_text().
    Expects a dict like:
    {
        "text": "...",
        "entities": [
            {"start": 0, "end": 12, "label": "ORG"},
            {"start": 35, "end": 39, "label": "PROJECT"}
        ]
    }
    Returns:
        {
            "tokens": [...],
            "labels": [...]
        }
    """
    global tokenizer
    global label_to_id

    text = entry.get("text")
    entities = entry.get("entities", [])
    if not text:
        print("⚠️ Skipping entry with no text.")
        return None

    # Tokenize with offsets to align character spans
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False,
    )
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    offsets = encoding["offset_mapping"]

    # Default all tokens to "O"
    labels = ["O"] * len(tokens)

    # Assign BIO labels based on entity spans
    for ent in entities:
        ent_start, ent_end, ent_label = ent["start"], ent["end"], ent["label"]
        for i, (start, end) in enumerate(offsets):
            if start is None or end is None:
                continue
            if start >= ent_start and end <= ent_end:
                prefix = "B-" if start == ent_start else "I-"
                label_tag = prefix + ent_label
                labels[i] = label_tag if label_tag in label_to_id else "O"

    return {"tokens": tokens, "labels": labels}


def add_labels_in_model_format(batch: dict) -> dict:
    global label_to_id
    global tokenizer
    """Add labels in model-ready format."""
    tokens_batch = batch["tokens"]   # list of token lists
    labels_batch = batch["labels"]   # list of label lists

    # Tokenize all examples together
    tokenized = tokenizer(
        tokens_batch,
        truncation=True,
        is_split_into_words=True,
        padding=False,   # dynamic padding later by collator
    )

    all_label_ids = []

    # For each example in the batch
    for i, labels in enumerate(labels_batch):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []

        for word_idx in word_ids:
            # ✅ Guard against None (special tokens) and out-of-range indices
            if word_idx is None or word_idx >= len(labels):
                label_ids.append(-100)
            else:
                label = labels[word_idx]
                label_ids.append(label_to_id.get(label, label_to_id["O"]))

        all_label_ids.append(label_ids)

    tokenized["labels"] = all_label_ids
    return tokenized


In [10]:
# Step 1. Smart chunking (handles long or short texts)
# this will handle long texts by smartly chunking them into smaller segments
chunked_dataset = preprocess_label_studio_data_for_long_text()

# Step 2. Convert each chunk to tokens + labels
examples = [
    process_one_label_studio_entry(entry) for entry in chunked_dataset
]
print("EXAMPLES" ,examples)
# examples would be a list of dictionaries with keys "tokens" and "labels"
# each item in the list is a chunk of the original text
examples = [ex for ex in examples if ex is not None]
dataset = Dataset.from_list(examples)
print(f"Prepared {len(examples)} examples from {len(chunked_dataset)} chunks")

print(examples[0]["tokens"])
print(examples[0]["labels"])



Preprocessing long Label Studio entries:   0%|          | 0/4 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Token indices sequence length is longer than the specified maximum sequence length for this model (2896 > 512). Running this sequence through the model will result in indexing errors
Preprocessing long Label Studio entries: 100%|██████████| 4/4 [00:00<00:00, 34.14it/s]

This model accepts maximum 512 as an input
We have 2896 tokens in our text
We will need to chunk the text into smaller segments
✅ Created 6 chunks from 4 documents
This means we have splite the 1 whole document into 6 chunks
This model accepts maximum 512 as an input
We have 5391 tokens in our text
We will need to chunk the text into smaller segments
✅ Created 17 chunks from 4 documents
This means we have splite the 1 whole document into 17 chunks
This model accepts maximum 512 as an input
We have 4161 tokens in our text
We will need to chunk the text into smaller segments
✅ Created 26 chunks from 4 documents
This means we have splite the 1 whole document into 26 chunks
This model accepts maximum 512 as an input
We have 4188 tokens in our text
We will need to chunk the text into smaller segments
✅ Created 35 chunks from 4 documents
This means we have splite the 1 whole document into 35 chunks
✅ In total, we have created 35 chunks from 4 documents
EXAMPLES [{'tokens': ['board', 'approve

In [11]:
tokenized_dataset = dataset.map(add_labels_in_model_format, batched=True)

Map: 100%|██████████| 35/35 [00:00<00:00, 1072.15 examples/s]


In [12]:
dataset[0]

{'tokens': ['board',
  'approved',
  '27',
  'memorandum',
  'june',
  '16',
  ',',
  '2022',
  'to',
  ':',
  'boston',
  'redevelopment',
  'authority',
  'd',
  '/',
  'b',
  '/',
  'a',
  'boston',
  'planning',
  '&',
  'development',
  'agency',
  '(',
  'b',
  '##p',
  '##d',
  '##a',
  ')',
  '1',
  'and',
  'james',
  'art',
  '##hur',
  'je',
  '##mis',
  '##on',
  'ii',
  ',',
  'director',
  'from',
  ':',
  'michael',
  'christopher',
  ',',
  'director',
  'of',
  'development',
  'review',
  'case',
  '##y',
  'hines',
  ',',
  'deputy',
  'director',
  'of',
  'development',
  'review',
  'quin',
  '##n',
  'val',
  '##cic',
  '##h',
  ',',
  'project',
  'assistant',
  'subject',
  ':',
  '1',
  '-',
  '4',
  'terrace',
  'place',
  ',',
  'mission',
  'hill',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  '_',
  

In [13]:
from transformers import DataCollatorForTokenClassification


tokenized_dataset = tokenized_dataset.remove_columns(
    [col for col in tokenized_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
val_dataset = split["test"]

batch_size = 8
epochs = 3
learning_rate=2e-5

# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)


# optimizer
optimizer = AdamW(ner_model.parameters(), lr=learning_rate)


if torch.backends.mps.is_available():
    print("Using MPS device")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("Using CUDA device")
    device = torch.device("cuda")
else:
    print("Using CPU device")
    device = torch.device("cpu")

ner_model.to(device)

Using MPS device


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [14]:
print(ner_model)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [15]:
def evaluate(loader: DataLoader, device: torch.device, ner_model) -> float:
    # the eval() would set the model to evaluation mode, making it not trainable
    ner_model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            batch = {
                k: v.to(device)
                for k, v in batch.items()
                if isinstance(v, torch.Tensor)
            }
            outputs = ner_model(**batch)
            loss = outputs.loss
            total_loss += loss.item()
    return total_loss / len(loader)

In [16]:
print(tokenized_dataset.features)

{'labels': List(Value('int64')), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [17]:

# train the model
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    ner_model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        batch = {
            k: v.to(device)
            for k, v in batch.items()
            if isinstance(v, torch.Tensor)
        }
        outputs = ner_model(**batch)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(train_loader)
    print(f"Average loss: {avg_loss:.4f}")

    # evaluate the model
    evaluate(val_loader, device, ner_model)


Epoch 1/3


Training: 100%|██████████| 4/4 [00:01<00:00,  2.31it/s]


Average loss: 0.0000


Evaluating: 100%|██████████| 1/1 [00:00<00:00,  4.17it/s]


Epoch 2/3


...

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.78it/s]


Epoch 3/3


Training: 100%|██████████| 4/4 [00:01<00:00,  2.62it/s]


Average loss: 0.0000


Evaluating: 100%|██████████| 1/1 [00:00<00:00,  9.74it/s]
